# The Tool System

A coding agent is only as useful as the things it can *do*. The LLM client from the [previous notebook](/notebooks/apps/cda/01-client.html) can talk to a model, but the model can only produce text. To read files, write code, search a codebase, or run shell commands, the model needs **tools** — functions it can request the runtime to execute on its behalf.

In this notebook, we build the tool system in three layers: (i) **base abstractions** (`Tool` ABC, `ToolResult`, `ToolKind`, `FileDiff`) that define the contract every tool must satisfy, (ii) **seven builtin tools** covering file I/O, search, and shell execution, and (iii) a **`ToolRegistry`** that dispatches tool calls by name, validates parameters, and returns uniform results. By the end, calling a tool looks like `await registry.invoke("read_file", {"path": "main.py"}, cwd=project_root)` — the agent loop never needs to know tool internals.

## Base Abstractions

**ToolKind.** Every tool belongs to one of five categories. The classification drives **approval logic**: `READ` tools are safe and execute without asking; everything else may require user confirmation depending on the `ApprovalPolicy` from [NB01](/notebooks/apps/cda/01-client.html).

In [ ]:
from notebooks.agent.tools.base import ToolKind

for kind in ToolKind:
    mutating = kind not in {ToolKind.READ}
    print(f"  {kind.value:10s}  mutating={mutating}")

**ToolResult.** Every tool execution returns a `ToolResult` — a uniform envelope carrying `success`, `output`, `error`, and optional `metadata`, `diff`, and `exit_code` fields. Two factory methods (`success_result`, `error_result`) handle the common cases. The `to_model_output()` method formats the result for the LLM conversation — on success it returns just the output; on failure it returns `f"Error: {self.error}\n\nOutput:\n{self.output}"` so the model sees both the error message and any partial output.

In [ ]:
from notebooks.agent.tools.base import ToolResult

ok = ToolResult.success_result("file contents here", metadata={"path": "/tmp/x.py"})
print(f"success={ok.success}")
print(f"model sees: {ok.to_model_output()[:50]}...")

err = ToolResult.error_result("File not found: /tmp/nope.py")
print(f"\nsuccess={err.success}")
print(f"model sees: {err.to_model_output()}")

**FileDiff.** Write tools (`write_file`, `edit`) capture before/after state as a `FileDiff`. The `to_diff()` method produces a unified diff string via `difflib.unified_diff` — used by the UI layer to show what changed before the user approves. For new files, `old_content` is empty and `is_new_file=True`; the diff shows `/dev/null` as the source.

In [ ]:
from pathlib import Path
from notebooks.agent.tools.base import FileDiff

diff = FileDiff(
    path=Path("/tmp/example.py"),
    old_content="def greet():\n    print('hello')\n",
    new_content="def greet(name: str):\n    print(f'hello {name}')\n",
)
print(diff.to_diff())

**Tool.** The abstract base class every tool implements. Subclasses set four class attributes (`name`, `description`, `kind`, `schema`) and implement one async method: `execute(invocation) → ToolResult`. The `schema` is a Pydantic `BaseModel` that does double duty: (i) parameter validation via `validate_params()`, and (ii) conversion to OpenAI function-calling JSON via `to_openai_schema()`. The `is_mutating()` and `get_confirmation()` methods support the approval flow we will wire up in [NB04](/notebooks/apps/cda/04-hardening.html).

In [ ]:
from notebooks.agent.config import Config
from notebooks.agent.tools.builtin.read_file import ReadFileTool

config = Config()
tool = ReadFileTool(config)

print(f"name:        {tool.name}")
print(f"kind:        {tool.kind.value}")
print(f"mutating:    {tool.is_mutating({})}")
print(f"description: {tool.description}")

The `to_openai_schema()` method converts the Pydantic model into the JSON format expected by the chat completions API. This is exactly what gets passed as `tools=[...]` to `chat_completion()` from NB01:

In [ ]:
import json

schema = tool.to_openai_schema()
print(json.dumps(schema, indent=2))

:::{.callout-note}
Each tool defines a Pydantic `BaseModel` as its `schema`. The `to_openai_schema()` method calls `pydantic.json_schema.model_json_schema()` to convert it to the JSON Schema format that OpenAI's function-calling API expects. Adding a new tool is just: define a Pydantic model for the parameters, implement `execute()`, register it.

:::

## Builtin Tools

The agent ships with seven tools in three groups: **navigation** (`list_dir`, `glob`, `grep`), **file I/O** (`read_file`, `write_file`, `edit`), and **execution** (`shell`). We demo each on a temporary workspace.

In [ ]:
import tempfile
from pathlib import Path
from notebooks.agent.tools.base import ToolInvocation

workspace = Path(tempfile.mkdtemp(prefix="agent_demo_"))

# Populate with a small Python project
(workspace / "src").mkdir()
(workspace / "src" / "main.py").write_text(
    'import sys\n\ndef main():\n    print("Hello from main!")\n    return 0\n\nif __name__ == "__main__":\n    sys.exit(main())\n'
)
(workspace / "src" / "utils.py").write_text(
    'def add(a: int, b: int) -> int:\n    """Add two numbers."""\n    return a + b\n\ndef multiply(a: int, b: int) -> int:\n    """Multiply two numbers."""\n    return a * b\n'
)
(workspace / "README.md").write_text("# Demo Project\n\nA small project for tool demos.\n")
(workspace / "pyproject.toml").write_text('[project]\nname = "demo"\nversion = "0.1.0"\n')

print(f"Workspace: {workspace}")

### Navigation: `list_dir`, `glob`, `grep`

These are all `READ` tools — safe, no side effects, no approval needed.

In [ ]:
from notebooks.agent.tools.builtin.list_dir import ListDirTool

list_dir = ListDirTool(config)
result = await list_dir.execute(ToolInvocation(params={"path": "."}, cwd=workspace))
print(result.output)

In [ ]:
from notebooks.agent.tools.builtin.glob import GlobTool

glob_tool = GlobTool(config)
result = await glob_tool.execute(ToolInvocation(params={"pattern": "**/*.py"}, cwd=workspace))
print(result.output)

In [ ]:
from notebooks.agent.tools.builtin.grep import GrepTool

grep = GrepTool(config)
result = await grep.execute(ToolInvocation(
    params={"pattern": "def \\w+", "path": ".", "include": "*.py"},
    cwd=workspace,
))
print(result.output)

The output format `file:line: content` mirrors ripgrep — the model is already familiar with it from training data.

### File I/O: `read_file`, `write_file`, `edit`

`read_file` is `READ`; `write_file` and `edit` are `WRITE` (mutating). The key design choice for `edit`: it does **search-and-replace** on exact strings rather than line-number-based patching, because LLMs are unreliable at counting line numbers but good at reproducing exact text spans.

In [ ]:
from notebooks.agent.tools.builtin.read_file import ReadFileTool

read_file = ReadFileTool(config)
result = await read_file.execute(ToolInvocation(params={"path": "src/main.py"}, cwd=workspace))
print(result.output)
print(f"\nmetadata: {result.metadata}")

The output is line-numbered (`lineno|content`), so the model can reference specific lines. The `offset` and `limit` parameters support partial reads of large files:

In [ ]:
result = await read_file.execute(ToolInvocation(
    params={"path": "src/main.py", "offset": 3, "limit": 3},
    cwd=workspace,
))
print(result.output)

In [ ]:
from notebooks.agent.tools.builtin.write_file import WriteFileTool

write_file = WriteFileTool(config)
result = await write_file.execute(ToolInvocation(
    params={
        "path": "src/config.py",
        "content": "# Configuration\n\nDEBUG = False\nMAX_RETRIES = 3\n",
    },
    cwd=workspace,
))
print(result.output)
print(result.diff.to_diff())

**EditTool.** The most important tool in the set — this is what the agent uses most when writing code. The `old_string` must match exactly (including whitespace) and be unique in the file:

In [ ]:
from notebooks.agent.tools.builtin.edit_file import EditTool

edit = EditTool(config)
result = await edit.execute(ToolInvocation(
    params={
        "path": "src/main.py",
        "old_string": "def main():",
        "new_string": "def main() -> int:",
    },
    cwd=workspace,
))
print(result.output)
print(result.diff.to_diff())

When the `old_string` matches multiple locations, the tool rejects the edit and asks for more context. This self-correcting feedback is important for agent reliability — the model learns to provide more surrounding lines:

In [ ]:
result = await edit.execute(ToolInvocation(
    params={
        "path": "src/utils.py",
        "old_string": "    return",
        "new_string": "    return int(",
    },
    cwd=workspace,
))
print(f"success={result.success}")
print(result.to_model_output())

### Execution: `shell`

The shell tool is `SHELL` kind (always mutating). Key safety features: (i) a blocklist of dangerous commands (`rm -rf /`, fork bombs, etc.), (ii) configurable timeout (default 120s, max 600s), and (iii) environment sanitization — the `ShellEnvironmentPolicy` from Config strips `*KEY*`, `*TOKEN*`, `*SECRET*` patterns so the agent cannot accidentally leak credentials through shell commands.

In [ ]:
from notebooks.agent.tools.builtin.shell import ShellTool

shell = ShellTool(config)
result = await shell.execute(ToolInvocation(
    params={"command": "python src/main.py"},
    cwd=workspace,
))
print(f"exit_code={result.exit_code}")
print(result.output)

The blocklist rejects destructive commands before execution. We verify that `rm -rf /` is refused:

In [ ]:
# Blocked command
result = await shell.execute(ToolInvocation(
    params={"command": "rm -rf /"},
    cwd=workspace,
))
print(f"success={result.success}")
print(result.to_model_output())

## Tool Registry

The `ToolRegistry` is the central dispatcher. The agent loop does not call tools directly — it calls `registry.invoke(name, params, cwd)`. The registry handles tool lookup, parameter validation against the Pydantic schema, and error wrapping. The `create_default_registry()` factory loads all seven builtins, and `get_schemas()` returns the OpenAI-format tool definitions that get passed to `chat_completion()`.

In [ ]:
from notebooks.agent.tools.registry import create_default_registry

registry = create_default_registry(config)

for tool in registry.get_tools():
    print(f"  {tool.name:12s}  kind={tool.kind.value:8s}  {tool.description[:60]}")

Invoking through the registry is the same interface the agent loop will use in [NB03](/notebooks/apps/cda/03-agent.html):

In [ ]:
result = await registry.invoke("grep", {"pattern": "import", "include": "*.py"}, cwd=workspace)
print(result.output)

schemas = registry.get_schemas()
print(f"\n{len(schemas)} tool schemas ready for chat_completion()")

The registry gracefully handles errors — unknown tool names and invalid parameters return `ToolResult` errors rather than raising exceptions:

In [ ]:
result = await registry.invoke("nonexistent", {}, cwd=workspace)
print(f"Unknown tool  → success={result.success}, error={result.error}")

result = await registry.invoke("read_file", {}, cwd=workspace)
print(f"Missing param → success={result.success}, error={result.error[:60]}...")

### Cleanup

In [ ]:
import shutil

shutil.rmtree(workspace)
print(f"Cleaned up {workspace}")

## Summary

We built the tool system in three layers:

| Module | What it provides |
|--------|------------------|
| `tools/base.py` | `Tool` ABC, `ToolResult`, `ToolKind`, `FileDiff`, `ToolInvocation`, `ToolConfirmation` — the contracts |
| `tools/builtin/` | 7 tools: `read_file`, `write_file`, `edit`, `shell`, `list_dir`, `grep`, `glob` |
| `tools/registry.py` | `ToolRegistry` — lookup, validation, dispatch; `create_default_registry()` |

Key design decisions:

- **Pydantic schemas do double duty** — parameter validation *and* OpenAI function-calling JSON generation.
- **`edit` uses exact-string matching**, not line numbers — LLMs are bad at counting but good at reproducing text.
- **`ToolKind` drives approval** — `READ` tools auto-execute; `WRITE`/`SHELL`/`NETWORK`/`MEMORY` may require confirmation.
- **Every tool returns `ToolResult`** — the agent loop never needs to know tool internals.

In the [next notebook](/notebooks/apps/cda/03-agent.html), we build the agent loop that wires the LLM client to the tool registry — the model thinks, calls tools, observes results, and thinks again until the task is done.